# Substitution Matrix Evaluation

This notebook evaluates amino acid substitution matrices based on their correlations with biochemical properties and codon distances.

The correlations are reported descriptively, without significance testing. The 190 pairs are built from 20 amino acids, each of which enters 19 of them, so they are not the independent observations a Spearman p-value assumes. The committed ranking is produced by `scripts/build_matrix_ranking.py`, which reproduces every committed row before it is allowed to write.

**Author:** KSS Project  
**Date:** 2025-01-19

## 1. Imports and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# Add src to path for main application modules
sys.path.insert(0, '../scripts')
sys.path.append('../src')
from mutation_score import get_mutational_scores, load_substitution_matrix

# Import evaluation utilities from local module
from amino_acid_utils import (
    CODON_TABLE,
    AMINO_ACIDS,
    AA_3_TO_1,
    compute_codon_distance,
    get_min_codon_distance,
    calculate_property_distances,
    generate_amino_acid_pairs,
    precompute_codon_distances
)

## 2. Configuration Parameters

In [ ]:
# Configuration
MATRICES_DIR = "../src/substitution_matrices"
PROPERTIES_FILE = '../scripts/amino_acid_properties.csv'
OUTPUT_DIR = "../results/matrix_evaluation"
TOP_N = 10

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Genetic Code and Amino Acid Data

In [ ]:
# Use standard genetic code from amino_acid_utils
print(f"Amino acids: {len(AMINO_ACIDS)}")
print(f"Amino acids: {AMINO_ACIDS}")

## 4. Helper Functions

In [ ]:
def format_correlation(corr):
    """Format a correlation for the ranking table."""
    return f"{corr:.3f}"

## 5. Load Amino Acid Properties

In [ ]:
# Load amino acid properties using mapping from amino_acid_utils
df_properties = pd.read_csv(PROPERTIES_FILE)
df_properties['AA_1letter'] = df_properties['AA'].map(AA_3_TO_1)
df_properties = df_properties.set_index('AA_1letter')
df_properties = df_properties.drop('AA', axis=1)

# Display properties
print("Amino acid properties:")
display(df_properties)

# Convert to dictionary format (using 1-letter codes)
amino_acid_properties = {col: df_properties[col].to_dict() for col in df_properties.columns}
property_names = list(amino_acid_properties.keys())

## 6. Compute Reference Distances

In [ ]:
# Generate all amino acid pairs using utility function
pairs = generate_amino_acid_pairs()
print(f"Total amino acid pairs: {len(pairs)}")

# Compute minimum codon distances using utility function
min_codon_distances = precompute_codon_distances()

# Display distribution
codon_dist_values = list(min_codon_distances.values())
print(f"\nMinimum codon distance distribution:")
print(pd.Series(codon_dist_values).value_counts().sort_index())

## 7. Load Available Matrices

In [ ]:
# Find all matrix files
matrix_files = {}

for file in os.listdir(MATRICES_DIR):
    if file.endswith('.json'):
        matrix_files[file[:-5]] = os.path.join(MATRICES_DIR, file)

matrix_names = sorted(matrix_files.keys())
print(f"Available matrices: {len(matrix_names)}")
for name in matrix_names:
    print(f"  - {name}")


## 8. Evaluate Matrices

In [ ]:
def evaluate_matrix(matrix_name):
    """Evaluate a single substitution matrix."""
    # Get mutation scores
    scores = get_mutational_scores(pairs, substitution_matrix_type=matrix_name)
    scores_array = np.array([scores[p] for p in pairs])
    
    # Calculate correlations with properties
    correlations = {}
    
    for prop_name in property_names:
        prop_distances = calculate_property_distances(amino_acid_properties[prop_name], pairs)
        correlations[prop_name] = stats.spearmanr(prop_distances, scores_array).statistic
    
    # Calculate correlation with codon distances
    codon_dist_array = np.array([min_codon_distances[p] for p in pairs])
    correlations['MinCodD'] = stats.spearmanr(codon_dist_array, scores_array).statistic
    
    # Composite score
    composite_score = sum(correlations.values())
    
    return correlations, composite_score


# Evaluate all matrices
results = {}
print("Evaluating matrices...\n")

for i, matrix_name in enumerate(matrix_names, 1):
    try:
        correlations, composite = evaluate_matrix(matrix_name)
        results[matrix_name] = {
            'correlations': correlations,
            'composite_score': composite
        }
        print(f"[{i}/{len(matrix_names)}] {matrix_name}: {composite:.3f}")
    except Exception as e:
        print(f"[{i}/{len(matrix_names)}] {matrix_name}: ERROR - {e}")

print(f"\nSuccessfully evaluated: {len(results)}/{len(matrix_names)}")

## 9. Create Results Table

In [ ]:
# Sort by composite score
sorted_matrices = sorted(results.items(), key=lambda x: x[1]['composite_score'], reverse=True)

# Build results dataframe
rows = []
for rank, (matrix_name, data) in enumerate(sorted_matrices, 1):
    row = {'Rank': rank, 'Matrix': matrix_name}
    
    # Add formatted correlations
    for prop in property_names + ['MinCodD']:
        row[prop] = format_correlation(data['correlations'][prop])
    
    row['Composite'] = round(data['composite_score'], 3)
    rows.append(row)

df_results = pd.DataFrame(rows)

# Display top N
print(f"\n{'='*120}")
print(f"SUBSTITUTION MATRICES RANKING")
print(f"{'='*120}\n")
display(df_results.head(len(df_results)).style.hide(axis='index'))

## 10. Best Matrix Details

In [ ]:
# Get best matrix
best_matrix_name = sorted_matrices[0][0]
best_data = sorted_matrices[0][1]

print(f"Best performing matrix: {best_matrix_name}")
print(f"Composite score: {best_data['composite_score']:.3f}\n")

# Detailed correlations
print("Detailed correlations:\n")
detail_rows = []
for prop in property_names + ['MinCodD']:
    detail_rows.append({
        'Property': prop,
        'Correlation': round(best_data['correlations'][prop], 4)
    })

df_best = pd.DataFrame(detail_rows)
display(df_best)

## 11. Save Results

In [ ]:
# Save top N matrices
top_file = os.path.join(OUTPUT_DIR, f'substitution_matrices_ranking_top_{TOP_N}.csv')
df_results.head(TOP_N).to_csv(top_file, index=False)
print(f"Top {TOP_N} results saved to: {top_file}")

# Save complete ranking
complete_file = os.path.join(OUTPUT_DIR, 'substitution_matrices_ranking_complete.csv')
df_results.to_csv(complete_file, index=False)
print(f"Complete results saved to: {complete_file}")

# Save best matrix details
best_file = os.path.join(OUTPUT_DIR, f'{best_matrix_name}_detailed_correlations.csv')
df_best.to_csv(best_file, index=False)
print(f"Best matrix details saved to: {best_file}")

## 12. Summary Statistics

In [ ]:
print("\nSUMMARY")
print("="*50)
print(f"Total matrices evaluated: {len(results)}")
print(f"Amino acid pairs analyzed: {len(pairs)}")
print(f"Properties analyzed: {len(property_names) + 1}")  # +1 for codon distance
print(f"\nBest matrix: {best_matrix_name}")
print(f"Best composite score: {best_data['composite_score']:.3f}")